Mostrando o caminho e verificando colunas

In [2]:
import pandas as pd
import mysql.connector
from mysql.connector import Error

caminho = r"C:\Users\felip\OneDrive\Área de Trabalho\sentirec\data\raw\Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv"
df = pd.read_csv(caminho)
df.columns


Index(['id', 'dateAdded', 'dateUpdated', 'name', 'asins', 'brand',
       'categories', 'primaryCategories', 'imageURLs', 'keys', 'manufacturer',
       'manufacturerNumber', 'reviews.date', 'reviews.dateAdded',
       'reviews.dateSeen', 'reviews.doRecommend', 'reviews.id',
       'reviews.numHelpful', 'reviews.rating', 'reviews.sourceURLs',
       'reviews.text', 'reviews.title', 'reviews.username', 'sourceURLs'],
      dtype='object')

Conectando ao Banco de Dados

In [3]:
try:
    conexao = mysql.connector.connect(
        host='localhost',
        user='root',
        password='#Fvaf221205', 
        database='db_sistemas_recomendacao'
    )
    cursor = conexao.cursor()
    print("Conexão bem-sucedida!")

except Error as e:
    print("Erro ao conectar:", e)


Conexão bem-sucedida!


Renomeando colunas e adicionando a coluna notas

In [ ]:
df = df[['name', 'categories', 'reviews.rating', 'reviews.text', 'reviews.date']]
df = df.dropna(subset=['reviews.rating', 'reviews.text'])

df = df.rename(columns={
    'name': 'nome_produto',
    'categories': 'categoria',
    'reviews.rating': 'nota',
    'reviews.text': 'texto',
    'reviews.date': 'data_avaliacao'
})

df['nota'] = df['nota'].astype(int)
df = df.reset_index(drop=True)
df.head()

,nome_produto,categoria,nota,texto,data_avaliacao
0,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...","Computers,Electronics Features,Tablets,Electro...",3,I thought it would be as big as small paper bu...,2017-09-03T00:00:00.000Z
1,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...","Computers,Electronics Features,Tablets,Electro...",5,This kindle is light and easy to use especiall...,2017-06-06T00:00:00.000Z
2,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...","Computers,Electronics Features,Tablets,Electro...",4,Didnt know how much i'd use a kindle so went f...,2018-04-20T00:00:00.000Z
3,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...","Computers,Electronics Features,Tablets,Electro...",5,I am 100 happy with my purchase. I caught it o...,2017-11-02T17:33:31.000Z
4,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...","Computers,Electronics Features,Tablets,Electro...",5,Solid entry level Kindle. Great for kids. Gift...,2018-04-24T00:00:00.000Z


Adicionando índice a coluna "id_usuario", removendo duplicatas das colunas id_usuario, nome_produto e categoria
Tipando o id_usuario pra int
Mostrando as primeiras colunas

In [5]:
df['id_usuario'] = df.index + 1

usuarios = df[['id_usuario']].drop_duplicates()
produtos = df[['nome_produto', 'categoria']].drop_duplicates()

usuarios['id_usuario'] = usuarios['id_usuario'].astype(int)
df.head()


,nome_produto,categoria,nota,texto,data_avaliacao,id_usuario
0,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...","Computers,Electronics Features,Tablets,Electro...",3,I thought it would be as big as small paper bu...,2017-09-03T00:00:00.000Z,1
1,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...","Computers,Electronics Features,Tablets,Electro...",5,This kindle is light and easy to use especiall...,2017-06-06T00:00:00.000Z,2
2,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...","Computers,Electronics Features,Tablets,Electro...",4,Didnt know how much i'd use a kindle so went f...,2018-04-20T00:00:00.000Z,3
3,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...","Computers,Electronics Features,Tablets,Electro...",5,I am 100 happy with my purchase. I caught it o...,2017-11-02T17:33:31.000Z,4
4,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...","Computers,Electronics Features,Tablets,Electro...",5,Solid entry level Kindle. Great for kids. Gift...,2018-04-24T00:00:00.000Z,5


Testando INSERT produtos

In [6]:
for _, linha in produtos.iterrows():
    cursor.execute('''
        INSERT IGNORE INTO tbl_produtos (nome_produto, categoria, preco)
        VALUES(%s, %s, %s)
    ''', (linha['nome_produto'], linha['categoria'], 0.0))
conexao.commit()
print("Produtos inseridos com sucesso.")

Produtos inseridos com sucesso.


Testando INSERT usuários

In [7]:
for _, linha in usuarios.iterrows():
    id_usuario = int(linha['id_usuario'])      
    nome = f"Usuário {id_usuario}"             
    email = f"user{id_usuario}@teste.com"    
    
    cursor.execute('''
        INSERT IGNORE INTO tbl_usuarios(id_usuario, nome_usuario, email)
        VALUES (%s, %s, %s)
    ''', (id_usuario, nome, email))

conexao.commit()
print("Usuários inseridos com sucesso.")

Usuários inseridos com sucesso.


Testando SELECT em produtos

In [8]:
cursor.execute("SELECT id_produto, nome_produto FROM tbl_produtos")
produto_dict = {nome: id for id, nome in cursor.fetchall()}
print(produto_dict)

{'Fone de Ouvido Bluetooth': 1, 'Livro - Python para Iniciantes': 2, 'Smartwatch Fitness': 3, 'Amazon Kindle E-Reader 6" Wifi (8th Generation, 2016)': 268, 'Amazon Echo Show Alexa-enabled Bluetooth Speaker with 7" Screen': 270, 'Amazon Fire TV with 4K Ultra HD and Alexa Voice Remote (Pendant Design) | Streaming Media Player': 271, 'Amazon - Echo Plus w/ Built-In Hub - Silver': 272, 'Amazon 9W PowerFast Official OEM USB Charger and Power Adapter for Fire Tablets and Kindle eReaders': 273, 'Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16 GB, Blue Kid-Proof Case': 274, 'Kindle E-reader - White, 6 Glare-Free Touchscreen Display, Wi-Fi - Includes Special Offers': 275, 'Fire Tablet, 7 Display, Wi-Fi, 16 GB - Includes Special Offers, Black': 276, 'Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16 GB, Green Kid-Proof Case': 277, 'All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Blue': 278, 'All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 32 GB - Includes Special Offer

Testando INSERT em avaliações e formatando datas

In [9]:
for _, linha in df.iterrows():
    try:
        id_usuario = int(linha['id_usuario'])
        id_produto = produto_dict.get(linha['nome_produto'])
        nota = float(linha['nota'])
        texto = str(linha['texto']).strip()

        data_raw = linha['data_avaliacao']
        data = pd.to_datetime(data_raw, errors='coerce')

        if pd.isna(data):
            raise ValueError("Data inválida")

        data = data.strftime('%Y-%m-%d %H:%M:%S')

        if id_produto:
            cursor.execute("""
                INSERT INTO tbl_avaliacoes (id_usuario, id_produto, nota, texto, data_avaliacao)
                VALUES (%s, %s, %s, %s, %s)
            """, (id_usuario, id_produto, nota, texto, data))

    except Exception as e:
        print(f"Erro na linha {linha.name}: {e}")

conexao.commit()
print("Avaliações inseridas com sucesso.")

Avaliações inseridas com sucesso.


Encerrando conexão

In [10]:
cursor.close()
conexao.close()
print("Conexão encerrada.")

Conexão encerrada.
